# AdaptiveSLM Training on Colab

Trains the AdaptiveSLM on-device model end-to-end on a **Colab T4 GPU**.

**Pipeline:** pre-training (custom deep-thin **30-layer, 768-hidden, GQA, RoPE, tied-embedding** model, ~305M params) → knowledge distillation from **Qwen3-4B-Instruct-2507** → **PAKD** profile-aware fine-tuning → **elastic-depth HF export** (30L/22L/15L) → **GGUF quantization**.

**Architecture notes**

- Modern dense SLM recipe: **MobileLLM-style deep-thin** + **Qwen3-style RoPE/GQA**.
- **MoE is deliberately NOT used** at sub-512MB scale — MoE only pays off at ≥1B active params (Qwen3 itself ships dense models below 4B).
- Elasticity comes from **MatFormer-style nested depth** with aux exit heads at layers 15/22, tied to PAKD user profiles — every depth prefix (30L/22L/15L) is a valid standalone model.

**Hardware:** T4 GPU required · **Total time:** ~6–12h

In [ ]:
# Install dependencies.
# NOTE: torch is preinstalled on Colab — NEVER reinstall it (it breaks the CUDA wheels).
!pip install -q transformers datasets accelerate bitsandbytes safetensors sentencepiece gguf tqdm pyyaml psutil 2>&1 | tail -3

In [ ]:
# GPU check — this pipeline requires a T4 GPU (or better)
import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected — the training pipeline cannot run on CPU.")
    print("Fix: Runtime → Change runtime type → Hardware accelerator: T4 GPU, then re-run from the top.")
    raise RuntimeError("No GPU available — switch to a GPU runtime before continuing.")

print(f"Device:       {torch.cuda.get_device_name(0)}")
print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"bf16 support: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Clone the repo (idempotent — safe to re-run, even after %cd has moved us into training/)
import os

if os.path.exists('train.py'):
    # Already inside Adaptive-SLM/training (cell re-run) — just update the repo
    !git -C .. pull 2>&1 | tail -3
elif os.path.exists('Adaptive-SLM'):
    !git -C Adaptive-SLM pull 2>&1 | tail -3
    %cd Adaptive-SLM/training
else:
    !git clone https://github.com/Anubhavera/Adaptive-SLM.git 2>&1 | tail -3
    %cd Adaptive-SLM/training

In [ ]:
# Prepare training data -> data/pretrain_corpus.jsonl, data/distill_data.jsonl, data/pakd_data.jsonl
!python prepare_cloud_data.py --phase all --output-dir ./data --pretrain-samples 100000 --distill-samples 50000 --pakd-samples 20000 2>&1 | tail -20

### Training phases (cells 5–8)

Run cells 5→8 in order; checkpoints chain automatically; each phase resumes the previous one (every phase auto-loads the latest checkpoint from `./checkpoints`).

| # | Phase | Data | Output | ~T4 time |
|---|-------|------|--------|----------|
| 5 | pretrain | `data/pretrain_corpus.jsonl` | `checkpoints/pretrained.pt` | 2–4h |
| 6 | distill | `data/distill_data.jsonl` | `checkpoints/distilled.pt` | 1–2h |
| 7 | pakd | `data/pakd_data.jsonl` | `checkpoints/pakd_finetuned.pt` | ~1h |
| 8 | export | — | `checkpoints/hf_export/depth{30,22,15}` + `../models/*.gguf` | ~15min |

In [ ]:
# Phase 1: Pre-training from scratch (~2-4h on T4)
!python train.py --phase pretrain --data-dir ./data 2>&1 | tee pretrain.log | tail -5

In [ ]:
# Phase 2: Knowledge distillation from Qwen3-4B-Instruct-2507 (4-bit teacher, auto-loaded) (~1-2h)
!python train.py --phase distill --data-dir ./data 2>&1 | tee distill.log | tail -5

In [ ]:
# Phase 3: PAKD profile-aware fine-tuning — the novel contribution (~1h)
!python train.py --phase pakd --data-dir ./data 2>&1 | tee pakd.log | tail -5

In [ ]:
# Export to HuggingFace format (elastic depths 30/22/15)
!python train.py --phase export --output-dir ./checkpoints --export-depths 30,22,15 2>&1 | tail -10

# Convert to GGUF + quantize to Q4_K_M via llama.cpp (CMake build)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp 2>&1 | tail -1
!cmake -S llama.cpp -B llama.cpp/build 2>&1 | tail -2
!cmake --build llama.cpp/build --target llama-quantize -- -j2 2>&1 | tail -2
!mkdir -p ../models
!python llama.cpp/convert_hf_to_gguf.py checkpoints/hf_export/depth30 --outfile ../models/adaptive_slm-30l-f16.gguf 2>&1 | tail -3
!./llama.cpp/build/bin/llama-quantize ../models/adaptive_slm-30l-f16.gguf ../models/adaptive_slm-30l-q4_k_m.gguf Q4_K_M 2>&1 | tail -5

### Smoke test

Sanity-check the exported model with transformers before quantization.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("checkpoints/hf_export/depth30")
model = AutoModelForCausalLM.from_pretrained("checkpoints/hf_export/depth30", dtype=torch.float16, device_map="cuda")
model.eval()

messages = [{"role": "user", "content": "Explain photosynthesis in one sentence."}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
inputs = {k: v.to("cuda") for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Download the artifacts

Download the .gguf files from `../models/` (Files sidebar → right-click → Download). Expected sizes: 30L ≈ 190MB, 22L ≈ 145MB, 15L ≈ 100MB at Q4_K_M.

To also quantize the elastic 22L/15L siblings, re-run the last two commands of the export cell against `checkpoints/hf_export/depth22` and `checkpoints/hf_export/depth15` (both are already exported by `--export-depths 30,22,15`).